# 253 — Recon pipeline debug + validation

End-to-end audit of the electrode reconstruction pipeline. Diagnoses why
specific patients (EL043, EL044, or anything you put in `FOCUS_PATIENTS`)
don't end up with electrodes in MOBA's 3D brain, and validates the pipeline
as a whole.

**Pipeline stages we audit:**

1. **Per-patient fsaverage coord CSVs** — `outputs/250_recon/fsaverage/coords/<PID>_contacts_fsaverage.csv`
   (the OUTPUT of recon, in MNI/fsaverage space; what 252 + 211 read from).
2. **Per-patient raw recon dirs** — `outputs/250_recon/<PID>/` (glassbrain, etc.)
3. **Aparc cache** — `outputs/250_recon/fsaverage/aparc_lookup.csv` (211 output).
4. **Latest clustering run's `labels.csv`** — what patients are in the canonical sample set?
5. **Per-run recon CSVs** — `<run_dir>/recon/<algo>_<run_id>__with_fsaverage.csv` (252 output, what MOBA fetches).

**Diagnostics produced:**

* §1–5 Per-source inventory tables (patient counts, electrode counts, schema check).
* §6 Coverage matrix — patient × pipeline-stage boolean grid (the headline diagnostic).
* §7 Naming variant detection — case, suffix, padding mismatches across sources.
* §8 Deep dive on `FOCUS_PATIENTS` — searches every file for any reference.
* §9 Per-patient electrode name comparison between labels.csv and coord CSV (round-trip test).
* §10 MNI scatter visualisation — confirm projection landed where expected.
* §11 Verdict table — per focus patient: diagnosis + suggested action.

Read-only — never modifies any artifact. Safe to re-run anytime.

## 0 — Imports + config

In [1]:
import os, sys, json, re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_cluster_run as R

CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
RECON_ROOT     = CLUSTERING_DIR.parent / '250_recon'
FSAV_DIR       = RECON_ROOT / 'fsaverage'
COORDS_DIR     = FSAV_DIR / 'coords'
APARC_CACHE    = FSAV_DIR / 'aparc_lookup.csv'
INDEX_PATH     = CLUSTERING_DIR / 'index.json'

# Patients to deep-dive on. Anything here gets per-file + per-stage drill-down
# in §8 and a row in the §11 verdict table.
FOCUS_PATIENTS = ['EL043', 'EL044']

# Patient ID prefixes recognized by the pipeline. Anything in labels.csv that
# doesn't match one of these is reported as 'unknown cohort' in §7.
KNOWN_COHORT_PREFIXES = ('EL', 'PAT_', 'MicroEPI', 'G-', 'B-')

# Pretty-print options for the big tables
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print(f'RECON_ROOT  = {RECON_ROOT}  (exists={RECON_ROOT.exists()})')
print(f'COORDS_DIR  = {COORDS_DIR}  (exists={COORDS_DIR.exists()})')
print(f'APARC_CACHE = {APARC_CACHE}  (exists={APARC_CACHE.exists()})')
print(f'FOCUS_PATIENTS = {FOCUS_PATIENTS}')

RECON_ROOT  = \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon  (exists=True)
COORDS_DIR  = \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\coords  (exists=True)
APARC_CACHE = \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\aparc_lookup.csv  (exists=True)
FOCUS_PATIENTS = ['EL043', 'EL044']


## §1 — Per-patient fsaverage coord CSVs

Inventories every `<PID>_contacts_fsaverage.csv` in `coords/`. For each file:
* Parses the filename to get the expected PID
* Reads the file, lists actual columns + row count
* Compares the `patient` column's unique values against the filename PID
  (if these disagree, the file is mislabeled)
* Records the schema (column set) so we can flag inconsistent schemas
  across patients (e.g. some files have `cohort` / `name_raw` / `is_wm`, others don't)

In [2]:
coord_files = sorted(COORDS_DIR.glob('*_contacts_fsaverage.csv'))
# Filter out aggregate files (e.g. ALL_PATIENTS_*)
coord_files = [p for p in coord_files if not p.name.upper().startswith('ALL_PATIENTS')]

rows = []
schemas = defaultdict(list)
coord_csvs_by_pid = {}

for p in coord_files:
    pid_from_filename = p.name.replace('_contacts_fsaverage.csv', '')
    try:
        df = pd.read_csv(p)
        cols = tuple(df.columns)
        schemas[cols].append(pid_from_filename)
        coord_csvs_by_pid[pid_from_filename] = df
        pid_in_col = sorted(df['patient'].astype(str).unique().tolist()) if 'patient' in df.columns else ['<no patient col>']
        mismatch = pid_from_filename not in pid_in_col and pid_in_col != ['<no patient col>']
        hemi_counts = (df['hemi'].astype(str).value_counts().to_dict()
                        if 'hemi' in df.columns else {})
        rows.append({
            'pid_from_filename': pid_from_filename,
            'pid_in_column':     ','.join(pid_in_col),
            'mismatch':          mismatch,
            'n_rows':            len(df),
            'n_unique_names':    df['name'].nunique() if 'name' in df.columns else None,
            'hemi_counts':       hemi_counts,
            'has_cohort':        'cohort' in df.columns,
            'has_name_raw':      'name_raw' in df.columns,
            'has_is_wm':         'is_wm' in df.columns,
        })
    except Exception as e:
        rows.append({
            'pid_from_filename': pid_from_filename,
            'pid_in_column':     f'<read error: {type(e).__name__}>',
            'mismatch':          True,
            'n_rows':            None,
        })

df_coord_inv = pd.DataFrame(rows).sort_values('pid_from_filename').reset_index(drop=True)
print(f'Found {len(coord_files)} per-patient coord CSVs.\n')
print('--- Coord CSV inventory ---')
print(df_coord_inv.to_string(index=False))

print(f'\n--- Schema variants ({len(schemas)} distinct column sets) ---')
for cols, pids in schemas.items():
    print(f'  cols={list(cols)}')
    print(f'  patients ({len(pids)}): {pids}')

PIDS_WITH_COORD_CSV = set(df_coord_inv['pid_from_filename'].tolist())

Found 27 per-patient coord CSVs.

--- Coord CSV inventory ---
pid_from_filename pid_in_column  mismatch  n_rows  n_unique_names          hemi_counts  has_cohort  has_name_raw  has_is_wm
            EL030         EL030     False     102             102   {'R': 75, 'L': 27}        True          True       True
            EL033         EL033     False      92              92   {'R': 68, 'L': 24}        True          True       True
            EL034         EL034     False     105             105   {'L': 90, 'R': 15}        True          True       True
            EL035         EL035     False     173             173  {'R': 149, 'L': 24}        True          True       True
            EL036         EL036     False      71              71    {'R': 63, 'L': 8}        True          True       True
            EL037         EL037     False     156             156  {'L': 120, 'R': 36}        True          True       True
            EL038         EL038     False     166             166  {'R

## §2 — Per-patient raw recon dirs

Lists every subdir under `outputs/250_recon/` except `fsaverage/`. These are
the upstream of `coords/*_contacts_fsaverage.csv` — if a patient has a recon
dir but no fsaverage CSV, the projection step failed.

In [3]:
rows = []
raw_dirs = []
for d in sorted(RECON_ROOT.iterdir()) if RECON_ROOT.exists() else []:
    if d.is_dir() and d.name != 'fsaverage':
        raw_dirs.append(d)
        # Try to characterize what's inside
        children = sorted(d.iterdir()) if d.exists() else []
        child_names = [c.name for c in children]
        rows.append({
            'pid':       d.name,
            'n_children': len(child_names),
            'has_glassbrain': 'glassbrain' in child_names,
            'children':  ','.join(child_names[:6]) + ('...' if len(child_names) > 6 else ''),
        })

df_raw_inv = pd.DataFrame(rows).sort_values('pid').reset_index(drop=True) if rows else pd.DataFrame()
print(f'Found {len(raw_dirs)} per-patient raw recon dirs under {RECON_ROOT}.\n')
if len(df_raw_inv):
    print(df_raw_inv.to_string(index=False))
PIDS_WITH_RAW_DIR = set(df_raw_inv['pid'].tolist()) if len(df_raw_inv) else set()

Found 21 per-patient raw recon dirs under \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon.

             pid  n_children  has_glassbrain                 children
           EL033           1            True               glassbrain
           EL034           1            True               glassbrain
           EL035           1            True               glassbrain
           EL036           1            True               glassbrain
           EL038           1            True               glassbrain
           EL039           1            True               glassbrain
           EL040           1            True               glassbrain
           EL042           1            True               glassbrain
           EL043           1            True               glassbrain
           EL045           1            True               glassbrain
        PAT_2868           1            True               glassbrain
        PAT

## §3 — Aparc cache coverage (211 output)

Per-patient electrode counts in `aparc_lookup.csv`. If a patient has a coord
CSV but isn't in aparc, `build_aparc_cache` either skipped them or errored.

In [13]:
pd.DataFrame({
        'n_electrodes': g.size(),
        'n_unknown':    g['aparc_label'].apply(lambda s: (s == 'unknown').sum()),
        'n_distinct_regions': g['aparc_label'].nunique(),
        'hemi_counts':  g['hemi'].apply(lambda s: dict(s.value_counts())),
    })

,n_electrodes,n_unknown,n_distinct_regions,hemi_counts
"(EL030, lh)",NaN,NaN,NaN,27.0
"(EL030, rh)",NaN,NaN,NaN,75.0
"(EL033, lh)",NaN,NaN,NaN,24.0
"(EL033, rh)",NaN,NaN,NaN,68.0
"(EL034, lh)",NaN,NaN,NaN,90.0
"(EL034, rh)",NaN,NaN,NaN,15.0
"(EL035, lh)",NaN,NaN,NaN,24.0
"(EL035, rh)",NaN,NaN,NaN,149.0
"(EL036, lh)",NaN,NaN,NaN,8.0
"(EL036, rh)",NaN,NaN,NaN,63.0


In [4]:
if APARC_CACHE.exists():
    df_aparc = pd.read_csv(APARC_CACHE)
    print(f'aparc cache: {len(df_aparc)} electrodes total\n')
    print('--- Per-patient counts ---')
    g = df_aparc.groupby('patient', dropna=False)
    df_aparc_per_pat = pd.DataFrame({
        'n_electrodes': g.size(),
        'n_unknown':    g['aparc_label'].apply(lambda s: (s == 'unknown').sum()),
        'n_distinct_regions': g['aparc_label'].nunique(),
        'hemi_counts':  g['hemi'].apply(lambda s: dict(s.value_counts())),
    }).reset_index().rename(columns={'patient': 'pid'}).sort_values('pid')
    print(df_aparc_per_pat.to_string(index=False))
    PIDS_IN_APARC = set(df_aparc_per_pat['pid'].astype(str).tolist())
else:
    df_aparc = None
    PIDS_IN_APARC = set()
    print(f'[missing] {APARC_CACHE}\nRun 211_validation.ipynb (Section C: aparc cache) to build.')

aparc cache: 3953 electrodes total

--- Per-patient counts ---


KeyError: 'pid'

## §4 — Patients in `labels.csv` (latest run per feature_set)

Picks the most recent run for each (method, feature_set) and lists the unique
patient_ids in that run's `labels.csv`. These are the patients the clustering
pipeline actually uses — if a patient is in `labels.csv` but missing from coords
(§1) or aparc (§3), MOBA's brain pane shows their cluster but no electrode dots.

In [5]:
with open(INDEX_PATH) as f:
    INDEX = json.load(f)

# Latest run per (method, feature_set)
latest = {}
for r in INDEX['runs']:
    key = (r['method'], r['feature_set'])
    if key not in latest or r['run_id'] > latest[key]['run_id']:
        latest[key] = r

rows = []
labels_pid_set_by_run = {}
for key, r in latest.items():
    run_dir = CLUSTERING_DIR / r['path']
    labels_csv = run_dir / 'labels.csv'
    if not labels_csv.exists():
        rows.append({'run': r['path'], 'n_samples': 0, 'n_patients': 0, 'patients': '<labels.csv missing>'})
        continue
    df = pd.read_csv(labels_csv, usecols=lambda c: c in ('patient_id', 'electrode'))
    pids = sorted(df['patient_id'].astype(str).unique().tolist())
    labels_pid_set_by_run[r['path']] = set(pids)
    rows.append({
        'run':         r['path'],
        'n_samples':   len(df),
        'n_patients':  len(pids),
        'n_electrodes': df.groupby('patient_id')['electrode'].nunique().sum() if 'electrode' in df.columns else None,
        'patients':    pids,
    })

df_labels_inv = pd.DataFrame(rows)
print(df_labels_inv.drop(columns=['patients']).to_string(index=False))

# Union of patients across ALL latest runs — that's the population we care about
PIDS_IN_LABELS = set()
for s in labels_pid_set_by_run.values():
    PIDS_IN_LABELS.update(s)
print(f'\nUnion of patients across {len(latest)} latest runs: {sorted(PIDS_IN_LABELS)}')

                                       run  n_samples  n_patients  n_electrodes
    hierarchical/blob/runs/20260523_104436       1538          23           925
      hierarchical/hg/runs/20260529_192435       1538          23           925
hierarchical/minus101/runs/20260523_104339       1538          23           925
     hierarchical/raw/runs/20260523_000404       1538          23           925
   hierarchical/rawds/runs/20260528_181931       1538          23           925
          kmeans/blob/runs/20260523_104429       1538          23           925
            kmeans/hg/runs/20260529_192417       1538          23           925
      kmeans/minus101/runs/20260523_102735       1538          23           925
           kmeans/raw/runs/20260522_234711       1538          23           925
         kmeans/rawds/runs/20260528_181913       1538          23           925

Union of patients across 10 latest runs: ['EL030', 'EL035', 'EL037', 'EL038', 'EL040', 'EL042', 'EL043', 'EL044', 'EL04

## §5 — Per-run recon CSVs (252 output, what MOBA fetches)

Each clustering run *should* have a `<run_dir>/recon/<algo>_<run_id>__with_fsaverage.csv`
produced by `252_clustering_recon.ipynb`. Walks every run, reports whether the
recon CSV exists and what patients it covers.

In [6]:
rows = []
recon_pid_sets = {}
for r in INDEX['runs']:
    run_dir = CLUSTERING_DIR / r['path']
    algo_tag = f"{r['method']}_{r['feature_set']}"
    recon_csv = run_dir / 'recon' / f'{algo_tag}_{r["run_id"]}__with_fsaverage.csv'
    if not recon_csv.exists():
        rows.append({
            'run': r['path'], 'recon_csv': '<missing>',
            'n_rows': 0, 'n_patients': 0, 'patients': [],
        })
        continue
    df = pd.read_csv(recon_csv, usecols=lambda c: c in ('patient_id', 'patient', 'name'))
    # Prefer the patient_id from labels (canonical) over the joined `patient` column
    pid_col = 'patient_id' if 'patient_id' in df.columns else ('patient' if 'patient' in df.columns else None)
    pids = sorted(df[pid_col].astype(str).unique().tolist()) if pid_col else []
    recon_pid_sets[r['path']] = set(pids)
    rows.append({
        'run': r['path'],
        'recon_csv': str(recon_csv.relative_to(CLUSTERING_DIR)),
        'n_rows': len(df),
        'n_patients': len(pids),
        'patients': pids,
    })

df_recon_inv = pd.DataFrame(rows)
n_missing = (df_recon_inv['recon_csv'] == '<missing>').sum()
n_with = len(df_recon_inv) - n_missing
print(f'Per-run recon CSVs: {n_with}/{len(df_recon_inv)} runs have one ({n_missing} missing — run 252)\n')
print(df_recon_inv.drop(columns=['patients']).to_string(index=False))

# Union of recon-csv patients
PIDS_IN_PER_RUN_RECON = set()
for s in recon_pid_sets.values():
    PIDS_IN_PER_RUN_RECON.update(s)
print(f'\nUnion of patients in per-run recon CSVs: {sorted(PIDS_IN_PER_RUN_RECON)}')

Per-run recon CSVs: 25/43 runs have one (18 missing — run 252)

                                       run                                                                                                  recon_csv  n_rows  n_patients
    hierarchical/blob/runs/20260521_163124                                                                                                  <missing>       0           0
    hierarchical/blob/runs/20260522_205440                                                                                                  <missing>       0           0
    hierarchical/blob/runs/20260522_213636                                                                                                  <missing>       0           0
    hierarchical/blob/runs/20260523_102742         hierarchical\blob\runs\20260523_102742\recon\hierarchical_blob_20260523_102742__with_fsaverage.csv    1538          23
    hierarchical/blob/runs/20260523_104436         hierarchical\blob\runs\20260523_104

## §6 — Coverage matrix (headline diagnostic)

Boolean grid: rows = patients we know about (from any source), columns = pipeline stages.
A patient that's ✓ in `labels` but ✗ in `coord_csv` is what causes MOBA's brain to show that
cluster's samples but no electrode dots. A patient ✓ in `coord_csv` but ✗ in `aparc` means 211 skipped them.

In [7]:
all_pids = sorted(
    PIDS_WITH_COORD_CSV
    | PIDS_WITH_RAW_DIR
    | PIDS_IN_APARC
    | PIDS_IN_LABELS
    | PIDS_IN_PER_RUN_RECON
    | set(FOCUS_PATIENTS),
    key=lambda s: (not s.upper().startswith('EL'), s)
)

def _to_check(b): return '✓' if b else '·'

rows = []
for pid in all_pids:
    rows.append({
        'pid':         pid,
        'raw_dir':     _to_check(pid in PIDS_WITH_RAW_DIR),
        'coord_csv':   _to_check(pid in PIDS_WITH_COORD_CSV),
        'aparc':       _to_check(pid in PIDS_IN_APARC),
        'labels':      _to_check(pid in PIDS_IN_LABELS),
        'per_run_recon': _to_check(pid in PIDS_IN_PER_RUN_RECON),
        'focus':       '★' if pid in FOCUS_PATIENTS else '',
    })
df_cov = pd.DataFrame(rows)
print('--- Coverage matrix ---')
print(df_cov.to_string(index=False))

# Highlight problem patients
in_labels_no_coord  = sorted(PIDS_IN_LABELS - PIDS_WITH_COORD_CSV)
in_labels_no_aparc  = sorted(PIDS_IN_LABELS - PIDS_IN_APARC)
in_coord_no_aparc   = sorted(PIDS_WITH_COORD_CSV - PIDS_IN_APARC)
in_labels_no_recon  = sorted(PIDS_IN_LABELS - PIDS_IN_PER_RUN_RECON)
in_coord_not_labels = sorted(PIDS_WITH_COORD_CSV - PIDS_IN_LABELS)

print(f'\n[!] In labels.csv but NO coord CSV   (MOBA brain misses them):    {in_labels_no_coord}')
print(f'[!] In labels.csv but NO aparc entry (Stats tab anatomy missing): {in_labels_no_aparc}')
print(f'[!] Coord CSV exists but NO aparc    (211 skipped them):          {in_coord_no_aparc}')
print(f'[!] In labels.csv but NO per-run recon CSV (252 join failed):     {in_labels_no_recon}')
print(f'[i] Coord CSV exists but NOT in labels (filtered by activity gate or never imported): {in_coord_not_labels}')

NameError: name 'PIDS_IN_APARC' is not defined

## §7 — Naming variant detection

Looks for case differences, suffix patterns (`_FU1`, `_v2`, etc.), and padding
differences (`EL43` vs `EL043`) across all patient ID columns. Anything not
matching `KNOWN_COHORT_PREFIXES` is flagged.

In [8]:
def _normalize_pid(pid):
    """Crude normalization: uppercase + remove non-alphanumerics + drop trailing version tag."""
    s = re.sub(r'[^A-Z0-9_]', '', pid.upper())
    s = re.sub(r'_(FU\d*|V\d+|FOLLOWUP\d*|REDO\d*)$', '', s)
    return s

# Bucket every observed PID by its normalized form. Anything with >1 raw form
# in the same bucket = naming inconsistency (e.g. EL43 + EL043 + el043).
buckets = defaultdict(set)
for src_name, src_set in [
    ('coord_csv_filename', PIDS_WITH_COORD_CSV),
    ('raw_dir',            PIDS_WITH_RAW_DIR),
    ('aparc',              PIDS_IN_APARC),
    ('labels',             PIDS_IN_LABELS),
    ('per_run_recon',      PIDS_IN_PER_RUN_RECON),
]:
    for pid in src_set:
        buckets[_normalize_pid(str(pid))].add((src_name, str(pid)))

variant_rows = []
for norm, observations in buckets.items():
    raw_forms = sorted({raw for _, raw in observations})
    if len(raw_forms) > 1:
        variant_rows.append({
            'normalized': norm,
            'n_variants': len(raw_forms),
            'variants':   ', '.join(raw_forms),
            'sources':    ', '.join(sorted({src for src, _ in observations})),
        })

if variant_rows:
    print('--- Naming variant collisions ---')
    print(pd.DataFrame(variant_rows).to_string(index=False))
else:
    print('No naming variant collisions found.')

# Cohort sniff — anything from labels that doesn't match known prefixes
unknown_cohort = [pid for pid in PIDS_IN_LABELS
                  if not any(pid.startswith(pfx) for pfx in KNOWN_COHORT_PREFIXES)]
if unknown_cohort:
    print(f'\n[!] Patients in labels.csv that don\'t match KNOWN_COHORT_PREFIXES={KNOWN_COHORT_PREFIXES}: {unknown_cohort}')
else:
    print(f'\nAll labels.csv patients match KNOWN_COHORT_PREFIXES={KNOWN_COHORT_PREFIXES}.')

NameError: name 'PIDS_IN_APARC' is not defined

## §8 — Deep dive on `FOCUS_PATIENTS`

For every patient in `FOCUS_PATIENTS`, finds every file under `outputs/`
that contains the PID (case-insensitive), shows file size + a head/tail snippet.

In [9]:
def _grep_pid_in_outputs(pid, *, root, max_results=50):
    """Walk `root` and return paths whose filename OR first few KB contains pid."""
    hits = []
    pid_lower = pid.lower()
    for p in root.rglob('*'):
        if not p.is_file():
            continue
        # Cheap filename check first
        if pid_lower in p.name.lower():
            hits.append((p, 'filename'))
            if len(hits) >= max_results: break
            continue
        # Only grep CSV / TSV / JSON / TXT content (skip binaries)
        if p.suffix.lower() not in ('.csv', '.tsv', '.json', '.txt'):
            continue
        # Read first 32 KB only
        try:
            head = p.read_text(encoding='utf-8', errors='ignore')[:32_000]
            if pid_lower in head.lower():
                hits.append((p, 'content'))
                if len(hits) >= max_results: break
        except Exception:
            pass
    return hits

for pid in FOCUS_PATIENTS:
    print(f'\n========================================')
    print(f'  FOCUS: {pid}')
    print(f'========================================')
    # Quick boolean from §6 sets
    print(f'  raw_dir       : {"YES" if pid in PIDS_WITH_RAW_DIR else "no"}')
    print(f'  coord_csv     : {"YES" if pid in PIDS_WITH_COORD_CSV else "no"}')
    print(f'  aparc         : {"YES" if pid in PIDS_IN_APARC else "no"}')
    print(f'  labels        : {"YES" if pid in PIDS_IN_LABELS else "no"}')
    print(f'  per_run_recon : {"YES" if pid in PIDS_IN_PER_RUN_RECON else "no"}')

    # If coord CSV exists, show columns + sample rows
    if pid in coord_csvs_by_pid:
        df_c = coord_csvs_by_pid[pid]
        print(f'\n  COORD CSV ({pid}_contacts_fsaverage.csv): {len(df_c)} rows, columns={list(df_c.columns)}')
        print('  HEAD:')
        print(df_c.head(3).to_string(index=False))
        print('  Patient col uniques:', sorted(df_c['patient'].astype(str).unique().tolist()) if 'patient' in df_c.columns else '<none>')

    # If aparc has them, show counts
    if df_aparc is not None and pid in PIDS_IN_APARC:
        sub = df_aparc[df_aparc['patient'].astype(str) == pid]
        print(f'\n  APARC entries: {len(sub)}')
        print(sub['aparc_label'].value_counts().head(5).to_string())

    # If labels has them, show electrode counts
    if pid in PIDS_IN_LABELS:
        for run_path, pid_set in labels_pid_set_by_run.items():
            if pid not in pid_set: continue
            labels_csv = CLUSTERING_DIR / run_path / 'labels.csv'
            df_l = pd.read_csv(labels_csv, usecols=lambda c: c in ('patient_id', 'electrode', 'condition'))
            sub = df_l[df_l['patient_id'] == pid]
            print(f'\n  LABELS in {run_path}: {len(sub)} samples, {sub["electrode"].nunique()} unique electrodes')
            print('  electrode samples:', sorted(sub['electrode'].unique().tolist())[:10], '...')
            break  # one example is enough

    # Filesystem grep
    print(f'\n  Filesystem grep under {CLUSTERING_DIR.parent}:')
    for p, why in _grep_pid_in_outputs(pid, root=CLUSTERING_DIR.parent, max_results=20):
        rel = p.relative_to(CLUSTERING_DIR.parent.parent) if p.is_relative_to(CLUSTERING_DIR.parent.parent) else p
        print(f'    [{why:8s}] {rel}  ({p.stat().st_size:,} bytes)')


  FOCUS: EL043
  raw_dir       : YES
  coord_csv     : YES


NameError: name 'PIDS_IN_APARC' is not defined

## §9 — Per-patient electrode name match (round-trip test)

For every patient with BOTH a coord CSV and entries in `labels.csv`, compare
the electrode-name sets:

* `coord.name` — the recon-side label (e.g. `A1`, `A2`)
* `labels.electrode` — the ERSP-side label (e.g. `A_L10`, `A_L11`)

If these don't overlap, 252's join silently drops everything. Reports per-patient
the symmetric difference.

In [10]:
# Pick one labels.csv to use as the labels-side reference (latest kmeans+raw,
# falling back to any latest run with patient_id+electrode columns)
ref_labels_csv = None
for r in sorted(INDEX['runs'], key=lambda x: x['run_id'], reverse=True):
    candidate = CLUSTERING_DIR / r['path'] / 'labels.csv'
    if candidate.exists():
        df_test = pd.read_csv(candidate, nrows=1)
        if 'patient_id' in df_test.columns and 'electrode' in df_test.columns:
            ref_labels_csv = candidate
            break

if ref_labels_csv is None:
    print('No suitable labels.csv found (need patient_id + electrode columns).')
else:
    print(f'Reference labels.csv: {ref_labels_csv}\n')
    df_labels = pd.read_csv(ref_labels_csv, usecols=lambda c: c in ('patient_id', 'electrode'))
    df_labels['patient_id'] = df_labels['patient_id'].astype(str)
    df_labels['electrode']  = df_labels['electrode'].astype(str)

    rows = []
    for pid in sorted(PIDS_IN_LABELS):
        labels_names = set(df_labels[df_labels['patient_id'] == pid]['electrode'].unique())
        if pid in coord_csvs_by_pid and 'name' in coord_csvs_by_pid[pid].columns:
            coord_names = set(coord_csvs_by_pid[pid]['name'].astype(str).unique())
        else:
            coord_names = set()
        only_labels = labels_names - coord_names
        only_coord  = coord_names - labels_names
        intersect   = labels_names & coord_names
        rows.append({
            'pid':                pid,
            'n_labels':           len(labels_names),
            'n_coord':            len(coord_names),
            'n_intersect':        len(intersect),
            'match_pct':          (len(intersect) / len(labels_names) * 100) if labels_names else 0.0,
            'only_in_labels(5)':  sorted(only_labels)[:5],
            'only_in_coord(5)':   sorted(only_coord)[:5],
        })
    df_match = pd.DataFrame(rows).sort_values('match_pct')
    print('--- Per-patient electrode name match (sorted by match %) ---')
    print(df_match.to_string(index=False))
    print('\nA match_pct < 100 means 252 joins will silently drop those electrodes.')
    print('A match_pct == 0 with non-zero n_labels and n_coord = name-format mismatch (e.g. A_L10 vs A1).')

Reference labels.csv: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\hg\runs\20260529_192435\labels.csv

--- Per-patient electrode name match (sorted by match %) ---
     pid  n_labels  n_coord  n_intersect  match_pct                          only_in_labels(5)                   only_in_coord(5)
   EL030        72      102            0        0.0         [A_L1, A_L10, A_L11, A_L12, A_L13]      [AL1, AL10, AL11, AL12, AL13]
   EL035        36      173            0        0.0          [A_L10, A_L11, A_L12, A_L7, A_L8]       [AL1, AL10, AL11, AL12, AL2]
   EL037        15      156            0        0.0 [A_R10, EntG_L11, EntG_L2, EntG_L3, aH_L3]       [AL1, AL10, AL11, AL12, AL2]
   EL038        51      166            0        0.0          [A_L1, A_L10, A_L11, A_L12, A_L2]      [AL1, AL10, AL11, AL12, AL13]
   EL040        28      246            0        0.0    [A_R12, FP_R1, OF_R1, PHG_R12, STG_R10]      [AR1, AR10, 

## §10 — MNI scatter visualisation

Three orthogonal projections (sagittal / coronal / axial) of every patient's
fsaverage-coord electrodes. Focus patients get an extra-size marker and a
patient-name annotation. If a focus patient has no coord CSV, this section
just won't draw their dots — confirms by absence.

In [ ]:
if not coord_csvs_by_pid:
    print('No coord CSVs to plot.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    proj = [('Sagittal (Y vs Z)', 'y', 'z'),
            ('Coronal (X vs Z)',  'x', 'z'),
            ('Axial (X vs Y)',    'x', 'y')]
    cmap = plt.get_cmap('tab20')
    pids_sorted = sorted(coord_csvs_by_pid.keys())
    color_by_pid = {pid: cmap(i % 20) for i, pid in enumerate(pids_sorted)}

    for ax, (title, c1, c2) in zip(axes, proj):
        for pid, df_c in coord_csvs_by_pid.items():
            if c1 not in df_c.columns or c2 not in df_c.columns:
                continue
            is_focus = pid in FOCUS_PATIENTS
            ax.scatter(
                df_c[c1], df_c[c2],
                s=24 if is_focus else 6,
                c=[color_by_pid[pid]],
                edgecolor='black' if is_focus else 'none',
                linewidth=0.6 if is_focus else 0,
                alpha=0.85 if is_focus else 0.55,
                label=pid if is_focus else None,
                zorder=3 if is_focus else 1,
            )
        ax.set_xlabel(c1.upper()); ax.set_ylabel(c2.upper())
        ax.set_title(title); ax.set_aspect('equal'); ax.grid(alpha=0.3)
        ax.axhline(0, color='gray', lw=0.4); ax.axvline(0, color='gray', lw=0.4)
    # Single legend showing only focus patients
    handles, labels_ = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels_, loc='upper center', ncol=len(handles), fontsize=9, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f'fsaverage electrode coords · {len(coord_csvs_by_pid)} patients · ★ = focus',
                  fontsize=11, y=1.04)
    fig.tight_layout()
    plt.show()

## §11 — Verdict + recommended actions per focus patient

Per-focus-patient diagnosis based on the coverage matrix + name-match results.

In [11]:
verdict_rows = []
for pid in FOCUS_PATIENTS:
    has_raw   = pid in PIDS_WITH_RAW_DIR
    has_coord = pid in PIDS_WITH_COORD_CSV
    has_aparc = pid in PIDS_IN_APARC
    in_labels = pid in PIDS_IN_LABELS
    in_recon  = pid in PIDS_IN_PER_RUN_RECON

    # Decision tree
    if not in_labels and not has_coord:
        diagnosis = 'Patient not in pipeline at all.'
        action    = ('Check whether ERSP files were generated for this patient. '
                     'If yes, re-run lf_dataset.prepare_dataset (cache may need invalidation).')
    elif in_labels and not has_coord:
        diagnosis = 'In labels.csv but no fsaverage coord CSV — recon never ran or output missing.'
        action    = ('Run the per-patient recon (functions.lf_recon_shared.export_patient_contacts_*) '
                     'for this patient. Verify that '
                     '<server>/SEEG_EXPERIMENTS_BERN/Reconstruction/<pid_lowercase>/ contains the '
                     'FreeSurfer subject + Lookup.xlsx.')
    elif has_coord and not in_labels:
        diagnosis = 'Reconned but not in any labels.csv — filtered by activity gate, or never imported.'
        action    = ('Check raw ERSP files exist. If they do, the high-activity gate dropped every '
                     'sample for this patient. Inspect prop_above_pos / prop_below_neg in '
                     'lf_dataset.prepare_dataset to confirm.')
    elif in_labels and has_coord and not in_recon:
        diagnosis = 'Both sides exist but per-run recon CSV is missing them — 252 silently dropped the join.'
        action    = ('Most likely cause: electrode-name format mismatch (e.g. labels.electrode = "A_L10" '
                     'vs coord.name = "A1"). See §9 above for the exact diff. Fix in lf_recon_shared.py '
                     'or rebuild 252 with a corrected name normalizer, then re-run 252.')
    elif in_labels and has_coord and in_recon and not has_aparc:
        diagnosis = 'Recon CSV has them, but aparc cache missing — 211 build_aparc_cache skipped them.'
        action    = ('Re-run 211 Section C with verbose=True; check for projection / annot read errors '
                     'specific to this patient.')
    else:
        diagnosis = 'Patient appears fully integrated. If MOBA still shows no electrodes, refresh the page (Pages CDN lag) or hard-reload to clear cached coords CSV.'
        action    = 'Try Cmd-Shift-R in MOBA, then re-check the brain pane.'

    verdict_rows.append({
        'focus':     pid,
        'stages':    f'raw:{int(has_raw)} coord:{int(has_coord)} aparc:{int(has_aparc)} labels:{int(in_labels)} per_run:{int(in_recon)}',
        'diagnosis': diagnosis,
        'action':    action,
    })

df_verdict = pd.DataFrame(verdict_rows)
print('=' * 100)
print('VERDICT — recommended next steps per focus patient')
print('=' * 100)
for _, row in df_verdict.iterrows():
    print(f"\n★ {row['focus']}  ({row['stages']})")
    print(f"  → {row['diagnosis']}")
    print(f"     {row['action']}")
print('\n' + '=' * 100)

NameError: name 'PIDS_IN_APARC' is not defined

## Done

This notebook is read-only — nothing was modified.

**Common findings + fixes:**

| Symptom | Where to look | Fix |
|---|---|---|
| Patient in `labels.csv` but missing from `coord_csv` | §6 row | Re-run the per-patient recon (`lf_recon_shared.export_patient_contacts_tkrras_and_mosaic`). |
| Both exist but `per_run_recon` is missing them | §9 match table | Electrode-name format mismatch. Fix the join in 252 + re-run. |
| Coord CSV exists but no aparc | §3 + §6 | Re-run 211 Section C with `verbose=True`; look for projection errors. |
| Schema differences across coord CSVs (e.g. some have `cohort`/`name_raw`/`is_wm`, some don't) | §1 schemas | Cosmetic — 211 + 252 only read `patient`, `name`, `x`, `y`, `z`, `hemi`. But the divergence suggests two different recon-export versions ran at different times. Re-run the older patients with the current pipeline to standardize. |
| Naming variant (e.g. `EL43` + `EL043`) | §7 | Pick one canonical form, rename files + columns to match. Then re-run 211 + 252. |

**Re-runs needed after a fix:**
1. The per-patient recon notebook (probably 250 / 251) — regenerates the missing fsaverage coord CSV.
2. `211_validation.ipynb` Section C — rebuilds `aparc_lookup.csv`.
3. `252_clustering_recon.ipynb` — rebuilds per-run recon CSVs that MOBA fetches.
4. Push + Pages redeploys (1–2 min).